In [1]:

from pathlib import Path
from google.colab import drive
import pandas as pd
import json

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")

if not (PROJECT / "results/IITD").is_dir():
    drive.mount("/content/drive5")

if not (PROJECT / "results/IITD").is_dir():
    raise FileNotFoundError("IITD project folder not found.")

PATHS = {
    "split": PROJECT / "results/IITD/notebook04_baseline_recognition/IITD_baseline_dataset_split.csv",
    "traditional": PROJECT / "results/IITD/notebook05_traditional_augmentation/IITD_traditional_augmented_metadata.csv",
    "gan": PROJECT / "results/IITD/notebook06_gan_augmentation/improved_128/IITD_GAN_synthetic_metadata.csv",
    "diffusion_summary": PROJECT / "results/IITD/notebook07_diffusion/IITD_diffusion_training_summary.json",
    "diffusion_samples": PROJECT / "results/IITD/notebook07_diffusion/IITD_DDPM_sample_metadata.json",
}

OUT_DIR = PROJECT / "results/IITD/notebook09_dataset_augmentation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in PATHS.items():
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")
    if path.is_file():
        print(f"  Size: {path.stat().st_size / 1024:.1f} KB")

if not PATHS["split"].is_file():
    raise FileNotFoundError("Baseline split CSV is required.")

split_df = pd.read_csv(PATHS["split"])
print("\nBaseline split counts:")
print(split_df["split"].value_counts())

print("\nOutput directory:", OUT_DIR)
print("Cell 1 complete. No images copied or models trained.")


Mounted at /content/drive5
split: FOUND
  Size: 660.8 KB
traditional: FOUND
  Size: 721.0 KB
gan: FOUND
  Size: 256.3 KB
diffusion_summary: FOUND
  Size: 0.7 KB
diffusion_samples: FOUND
  Size: 0.5 KB

Baseline split counts:
split
train         1526
test           436
validation     218
Name: count, dtype: int64

Output directory: /content/drive5/MyDrive/Ocular_Project/results/IITD/notebook09_dataset_augmentation
Cell 1 complete. No images copied or models trained.


In [2]:

import pandas as pd
import json
from pathlib import Path

# Load metadata
split_df = pd.read_csv(PATHS["split"])
trad_df = pd.read_csv(PATHS["traditional"])
gan_df = pd.read_csv(PATHS["gan"])

# 1. Original baseline split manifest
original = split_df.copy()
original["data_source"] = "original"
original["identity_label_verified"] = True
original.to_csv(OUT_DIR / "IITD_original_split_manifest.csv", index=False)

# 2. Traditional augmentation manifest
# Only training augmentations should be used for training experiments.
traditional = trad_df.copy()
traditional["data_source"] = "traditional"
traditional["identity_label_verified"] = True

if "split" in traditional.columns:
    # Preserve only training-set augmentations in this experiment manifest.
    traditional_train = traditional[traditional["split"].astype(str).str.lower() == "train"].copy()
else:
    traditional_train = traditional.iloc[0:0].copy()

traditional_train.to_csv(
    OUT_DIR / "IITD_traditional_train_manifest.csv", index=False
)

# 3. GAN augmentation manifest
gan = gan_df.copy()
gan["data_source"] = "gan"

# Do not treat proposed conditional labels as verified biometric identity.
gan["identity_label_verified"] = False
gan.to_csv(OUT_DIR / "IITD_gan_manifest.csv", index=False)

# 4. Diffusion samples manifest
diffusion_path = PATHS["diffusion_samples"]
diffusion_records = []

if diffusion_path.is_file():
    with open(diffusion_path) as f:
        diffusion_data = json.load(f)

    # Inspect the metadata format instead of assuming its structure.
    if isinstance(diffusion_data, list):
        diffusion_records = diffusion_data
    elif isinstance(diffusion_data, dict):
        for key in ["samples", "images", "generated_images", "records"]:
            if isinstance(diffusion_data.get(key), list):
                diffusion_records = diffusion_data[key]
                break

diffusion = pd.DataFrame(diffusion_records)

if not diffusion.empty:
    diffusion["data_source"] = "diffusion"
    diffusion["identity_label_verified"] = False
    diffusion["split"] = "unlabeled_synthetic"
else:
    # A sample grid is not equivalent to a folder of individual images.
    diffusion = pd.DataFrame([{
        "data_source": "diffusion",
        "split": "unlabeled_synthetic",
        "identity_label_verified": False,
        "note": "Notebook 07 metadata does not contain a recognized list of individual image records. Only the saved sample grid is available for this manifest."
    }])

diffusion.to_csv(OUT_DIR / "IITD_diffusion_manifest.csv", index=False)

# 5. Dataset inventory summary
summary = {
    "original_train_images": int((original["split"] == "train").sum()),
    "original_validation_images": int((original["split"] == "validation").sum()),
    "original_test_images": int((original["split"] == "test").sum()),
    "traditional_train_augmentations": int(len(traditional_train)),
    "gan_synthetic_records": int(len(gan)),
    "diffusion_manifest_records": int(len(diffusion)),
    "diffusion_records_are_individual_images": bool(
        not diffusion.empty and "note" not in diffusion.columns
    ),
    "validation_and_test_preserved": True,
    "gpu_required": False,
    "identity_label_policy": (
        "GAN labels are proposed labels and are not treated as verified identities. "
        "Diffusion samples are unlabeled unless identity is independently verified."
    )
}

with open(OUT_DIR / "IITD_dataset_augmentation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Dataset manifest generation complete.")
print(json.dumps(summary, indent=2))
print("\nOutput files:")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"{p.name} — {p.stat().st_size / 1024:.1f} KB")


Dataset manifest generation complete.
{
  "original_train_images": 1526,
  "original_validation_images": 218,
  "original_test_images": 436,
  "traditional_train_augmentations": 4578,
  "gan_synthetic_records": 1275,
  "diffusion_manifest_records": 1,
  "diffusion_records_are_individual_images": false,
  "validation_and_test_preserved": true,
  "gpu_required": false,
  "identity_label_policy": "GAN labels are proposed labels and are not treated as verified identities. Diffusion samples are unlabeled unless identity is independently verified."
}

Output files:
IITD_dataset_augmentation_summary.json — 0.5 KB
IITD_diffusion_manifest.csv — 0.2 KB
IITD_gan_manifest.csv — 268.8 KB
IITD_original_split_manifest.csv — 687.5 KB
IITD_traditional_train_manifest.csv — 797.0 KB


# Conclusion — Dataset Augmentation and Manifest Preparation

This notebook organized the IIT Delhi ocular dataset and the available augmentation outputs into separate experiment manifests for downstream evaluation.

## Dataset Inventory

- Original training set: 1,526 images.
- Original validation set: 218 images.
- Original test set: 436 images.
- Traditional training augmentations: 4,578 records.
- GAN synthetic records: 1,275.

The original validation and test splits were preserved and were not augmented in this notebook.

## Synthetic Data Labeling Policy

The GAN metadata contains proposed conditional labels. These labels are not considered proof that a generated image preserves the corresponding subject's biometric identity. Diffusion samples are treated as unlabeled because the current diffusion model does not provide verified identity assignments.

The diffusion manifest therefore records the availability limitation instead of representing the sample grid as a complete dataset of individual images.

## Limitations

This notebook prepares metadata and manifests; it does not establish synthetic-image quality, identity preservation, privacy protection, or improved recognition accuracy. Those questions require separate evaluation. The current diffusion output must also be organized into individually indexed image files before it can be incorporated into a reproducible image-level augmentation experiment.

## Conclusion

The original split and the available traditional and GAN augmentation records have been organized for subsequent experiments. The original validation and test sets remain unchanged. The manifests provide a clear starting point for downstream recognition experiments while documenting the unresolved identity-label and diffusion-dataset limitations.